# Project 5B: Flow Matching from Scratch!

## Setup environment

In [ ]:
# We recommend using these utils.
# https://google.github.io/mediapy/mediapy.html
# https://einops.rocks/
!pip install mediapy einops --quiet

In [ ]:
# Import essential modules. Feel free to add whatever you need.
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from torchvision.datasets import MNIST
from torchvision.transforms import ToTensor, Compose, Normalize
from tqdm.auto import tqdm

## Neural Network Resources

In this part, you will build and train a [UNet](https://arxiv.org/abs/1505.04597), which is more complex than the MLP you implemented in the NeRF project.  We provide all class definitions you may need (but feel free to add or modify them as necessary).  

Instead of asking ChatGPT to write everything for you, please consult the following resources when you get stuck — they will help you understand how and why things work under the hood.

- PyTorch Documentation — [`Conv2d`](https://docs.pytorch.org/docs/stable/generated/torch.nn.Conv2d.html), [`ConvTranspose2d`](https://docs.pytorch.org/docs/stable/generated/torch.nn.ConvTranspose2d.html), and [`AvgPool2d`](https://docs.pytorch.org/docs/stable/generated/torch.nn.AvgPool2d.html).
- PyTorch Documentation - [`torchvision.datasets.MNIST`](https://docs.pytorch.org/vision/main/generated/torchvision.datasets.MNIST.html), the dataset we gonna use, and [`torch.utils.data.DataLoader`](https://docs.pytorch.org/docs/stable/data.html), the off-the-shell dataloader we can directly use.
- PyTorch [tutorial](https://docs.pytorch.org/tutorials/beginner/blitz/cifar10_tutorial.html) on how to train a classifier on CIFAR10 dataset. The structure of your training code will be very similar to this one.

# Part 1: Training a Single-step Denoising UNet


# Part 1.1: Implementing the UNet

## Implementing Simple and Composed Ops

In [ ]:
class Conv(nn.Module):
    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.GELU()
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class DownConv(nn.Module):
    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.GELU()
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class UpConv(nn.Module):
    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.ConvTranspose2d(in_channels, out_channels, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.GELU()
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class Flatten(nn.Module):
    def __init__(self):
        super().__init__()
        self.pool = nn.AvgPool2d(7)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.pool(x)


class Unflatten(nn.Module):
    def __init__(self, in_channels: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.ConvTranspose2d(in_channels, in_channels, kernel_size=7, stride=1, padding=0),
            nn.BatchNorm2d(in_channels),
            nn.GELU()
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class ConvBlock(nn.Module):
    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()
        self.conv = Conv(in_channels, out_channels)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.conv(x)


class DownBlock(nn.Module):
    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()
        self.down = DownConv(in_channels, out_channels)
        self.conv = Conv(out_channels, out_channels)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        out = self.down(x)
        out = self.conv(out)
        return out


class UpBlock(nn.Module):
    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()

        self.up = UpConv(in_channels, out_channels)
        self.conv = Conv(out_channels * 2, out_channels)

    def forward(self, x: torch.Tensor, skip: torch.Tensor) -> torch.Tensor:
        x_up = self.up(x)
        concat = torch.cat([x_up, skip], dim=1)
        out = self.conv(concat)
        return out

## Implementing Unconditional UNet

In [ ]:
class UnconditionalUNet(nn.Module):
    def __init__(self, in_channels: int = 1, num_hiddens: int = 128):
        super().__init__()

        self.init_conv = Conv(in_channels, num_hiddens)

        self.down1 = DownBlock(num_hiddens, num_hiddens * 2)

        self.down2 = DownBlock(num_hiddens * 2, num_hiddens * 4)

        self.flatten = Flatten()

        self.unflatten = Unflatten(num_hiddens * 4)

        self.up1 = UpBlock(num_hiddens * 4, num_hiddens * 2)

        self.up2 = UpBlock(num_hiddens * 2, num_hiddens)

        self.out_conv = nn.Conv2d(num_hiddens, in_channels, kernel_size=3, padding=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        assert x.shape[-2:] == (28, 28), "Expect input shape to be (28, 28)."

        x1 = self.init_conv(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)

        x_flat = self.flatten(x3)
        x_unflat = self.unflatten(x_flat)


        x_up1 = self.up1(x_unflat, x3)

        x_up2 = self.up2(x_up1, x2)

        out = self.out_conv(x_up2)
        return out

# Part 1.2: Using the UNet to Train a Denoiser

In [ ]:
# Visualize images at different noisy level
def visualize_noise_levels():
    ds = MNIST(root='./data', train=False, download=True, transform=ToTensor())
    x, _ = ds[0]

    sigmas = [0.0, 0.2, 0.5, 0.8, 1.0]
    fig, axes = plt.subplots(1, len(sigmas), figsize=(15, 3))

    for i, sigma in enumerate(sigmas):
        noise = torch.randn_like(x)
        noisy_x = x + sigma * noise
        axes[i].imshow(noisy_x.squeeze(), cmap='gray')
        axes[i].set_title(f"$\sigma={sigma}$")
        axes[i].axis('off')
    plt.show()

visualize_noise_levels()

## Part 1.2.1: Training

For this part, we provide some structure code for training. It is very basic, so feel free to change them or add your code. In later section we won't provide any training or visualization structure code.

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Set your hyperparameters
batch_size = 256
learning_rate = 1e-4
noise_level = 0.5
hidden_dim = 128
num_epochs = 5

In [ ]:
# Define your datasets and dataloaders
transform = Compose([ToTensor()]) # MNIST is already [0,1]

train_dataset = MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = MNIST(root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

In [ ]:
# Define your model, optimizer, and loss
model = UnconditionalUNet(in_channels=1, num_hiddens=hidden_dim).to(device)
optimizer = optim.Adam(model.parameters(), lr=learning_rate)
criterion = nn.MSELoss()

In [ ]:
# The training loops
train_losses = []

print("Starting training...")
for epoch in range(num_epochs):
    model.train()
    batch_losses = []

    for i, (images, _) in enumerate(tqdm(train_loader, desc=f"Epoch {epoch+1}")):
        images = images.to(device)

        # Generate noise
        noise = torch.randn_like(images).to(device)
        noisy_images = images + noise_level * noise

        # Forward pass
        # The model tries to predict the original clean image (denoising)
        outputs = model(noisy_images)

        loss = criterion(outputs, images)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        batch_losses.append(loss.item())

    avg_loss = sum(batch_losses) / len(batch_losses)
    train_losses.extend(batch_losses) # Or append avg_loss for cleaner plot
    print(f"Epoch {epoch+1} Average Loss: {avg_loss:.4f}")

    # Visualization at Epoch 1 and 5
    if (epoch + 1) == 1 or (epoch + 1) == num_epochs:
        model.eval()
        with torch.no_grad():
            # Get samples from test set
            test_imgs = next(iter(test_loader))[0][:8].to(device)
            test_noise = torch.randn_like(test_imgs).to(device)
            test_noisy = test_imgs + noise_level * test_noise
            denoised = model(test_noisy)

            # Plot
            fig, axs = plt.subplots(3, 8, figsize=(16, 6))
            for k in range(8):
                axs[0, k].imshow(test_imgs[k].cpu().squeeze(), cmap='gray')
                axs[0, k].axis('off'); axs[0,0].set_ylabel("Clean")

                axs[1, k].imshow(test_noisy[k].cpu().squeeze(), cmap='gray')
                axs[1, k].axis('off'); axs[1,0].set_ylabel("Noisy")

                axs[2, k].imshow(denoised[k].cpu().squeeze(), cmap='gray')
                axs[2, k].axis('off'); axs[2,0].set_ylabel("Denoised")
            plt.suptitle(f"Results after Epoch {epoch+1}")
            plt.show()

## Part 1.2.2: Out-of-Distribution Testing

In [ ]:
# Visualize OOD testing
def ood_testing(model):
    model.eval()
    ds = MNIST(root='./data', train=False, transform=ToTensor())
    img, _ = ds[0]
    img = img.unsqueeze(0).to(device)

    sigmas = [0.1, 0.3, 0.5, 0.8, 1.2, 2.0]

    fig, axs = plt.subplots(2, len(sigmas), figsize=(15, 5))

    with torch.no_grad():
        for i, sigma in enumerate(sigmas):
            noise = torch.randn_like(img)
            noisy_img = img + sigma * noise
            pred = model(noisy_img)

            axs[0, i].imshow(noisy_img.cpu().squeeze(), cmap='gray')
            axs[0, i].set_title(f"Input $\sigma={sigma}$")
            axs[0, i].axis('off')

            axs[1, i].imshow(pred.cpu().squeeze(), cmap='gray')
            axs[1, i].set_title("Denoised")
            axs[1, i].axis('off')
    plt.show()

ood_testing(model)

## Part 1.2.3 Denoising Pure Noise

In [ ]:
# Train to denoise pure noise
# Re-initialize model
model_pure = UnconditionalUNet(in_channels=1, num_hiddens=hidden_dim).to(device)
optimizer_pure = optim.Adam(model_pure.parameters(), lr=learning_rate)
criterion = nn.MSELoss()

print("Training on Pure Noise...")
for epoch in range(num_epochs):
    model_pure.train()
    for i, (images, _) in enumerate(tqdm(train_loader)):
        images = images.to(device)

        # Input is pure Gaussian noise
        pure_noise = torch.randn_like(images).to(device)

        outputs = model_pure(pure_noise)
        loss = criterion(outputs, images) # Try to map noise -> digit

        optimizer_pure.zero_grad()
        loss.backward()
        optimizer_pure.step()

    # Visualize
    if (epoch + 1) in [1, 5]:
        model_pure.eval()
        with torch.no_grad():
            sample_noise = torch.randn(8, 1, 28, 28).to(device)
            gen_imgs = model_pure(sample_noise)

            fig, axs = plt.subplots(1, 8, figsize=(16, 2))
            for k in range(8):
                axs[k].imshow(gen_imgs[k].cpu().squeeze(), cmap='gray')
                axs[k].axis('off')
            plt.suptitle(f"Generated from Pure Noise (Epoch {epoch+1})")
            plt.show()

# Part 2: Flow Matching

# Part 2.1: Implementing a Time-conditioned UNet

In [ ]:
class FCBlock(nn.Module):
    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_channels, out_channels),
            nn.GELU(),
            nn.Linear(out_channels, out_channels)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (N, 1) or (N,) -> Output: (N, out_channels)
        if x.ndim == 1:
            x = x.unsqueeze(1)
        return self.net(x)


class TimeConditionalUNet(nn.Module):
    def __init__(self, in_channels: int = 1, num_classes: int = 10, num_hiddens: int = 64):
        super().__init__()

        # Layers (Same structure as Unconditional)
        self.init_conv = Conv(in_channels, num_hiddens)
        self.down1 = DownBlock(num_hiddens, num_hiddens * 2)
        self.down2 = DownBlock(num_hiddens * 2, num_hiddens * 4)
        self.flatten = Flatten()

        self.fc1_t = FCBlock(1, num_hiddens * 4) # For Unflatten output
        self.fc2_t = FCBlock(1, num_hiddens * 2) # For Up1 output

        self.unflatten = Unflatten(num_hiddens * 4)
        self.up1 = UpBlock(num_hiddens * 4, num_hiddens * 2)
        self.up2 = UpBlock(num_hiddens * 2, num_hiddens)
        self.out_conv = nn.Conv2d(num_hiddens, in_channels, kernel_size=3, padding=1)

    def forward(self, x: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        # t: (N,)

        # Encoding
        x1 = self.init_conv(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)

        x_flat = self.flatten(x3)

        # Time Embedding
        t_emb1 = self.fc1_t(t) # (N, 4D)
        t_emb2 = self.fc2_t(t) # (N, 2D)

        # Reshape embeddings for broadcasting (N, C, 1, 1)
        t_emb1 = t_emb1.view(-1, t_emb1.shape[1], 1, 1)
        t_emb2 = t_emb2.view(-1, t_emb2.shape[1], 1, 1)

        # Bottleneck & Modulation 1
        x_unflat = self.unflatten(x_flat)
        x_unflat = x_unflat * t_emb1 # Modulate

        # Decoding & Modulation 2
        x_up1 = self.up1(x_unflat, x3)
        x_up1 = x_up1 * t_emb2 # Modulate

        x_up2 = self.up2(x_up1, x2)

        out = self.out_conv(x_up2)
        return out

## Implementing the Forward and Reverse Process for Time-conditioned Denoising

In [ ]:
def time_fm_forward(unet: TimeConditionalUNet, x_1: torch.Tensor, num_ts: int) -> torch.Tensor:
    unet.train()
    device = x_1.device
    batch_size = x_1.shape[0]

    x_0 = torch.randn_like(x_1)

    t = torch.rand(batch_size, device=device)

    t_img = t.view(-1, 1, 1, 1)

    # Flow Matching Path: x_t = (1 - t) * x_0 + t * x_1
    x_t = (1 - t_img) * x_0 + t_img * x_1

    # velocity = d/dt (x_t) = x_1 - x_0
    v_t = x_1 - x_0

    v_pred = unet(x_t, t)

    loss = F.mse_loss(v_pred, v_t)
    return loss

In [ ]:
@torch.inference_mode()
def time_fm_sample(unet: TimeConditionalUNet, img_wh: tuple[int, int], num_ts: int, seed: int = 0) -> torch.Tensor:
    unet.eval()
    torch.manual_seed(seed)
    device = next(unet.parameters()).device

    B = 16
    x = torch.randn(B, 1, img_wh[0], img_wh[1], device=device)

    timesteps = torch.linspace(0, 1, num_ts, device=device)
    dt = 1.0 / (num_ts - 1) if num_ts > 1 else 0

    for i in range(len(timesteps) - 1):
        t_curr = timesteps[i]

        # Expand t for batch
        t_batch = torch.full((B,), t_curr, device=device)

        # Predict velocity
        v_pred = unet(x, t_batch)

        # Update x
        x = x + v_pred * dt

    return x.clamp(0, 1) # MNIST is [0, 1]

In [ ]:
class TimeConditionalFM(nn.Module):
    def __init__(
        self,
        unet: TimeConditionalUNet,
        num_ts: int = 50,
        img_hw: tuple[int, int] = (28, 28),
    ):
        super().__init__()

        self.unet = unet
        self.num_ts = num_ts
        self.img_hw = img_hw


    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: (N, C, H, W) input tensor.

        Returns:
            (,) diffusion loss.
        """
        return time_fm_forward(
            self.unet, x, self.num_ts
        )


    @torch.inference_mode()
    def sample(
        self,
        img_wh: tuple[int, int],
        seed: int = 0,
    ):
        return time_fm_sample(
            self.unet, img_wh, self.num_ts, seed
        )


# Part 2.2: Training the Time-conditioned UNet

In [ ]:
# Hyperparams
lr = 1e-2
num_epochs = 10
hidden_dim = 64
num_ts = 50 # Not used in training loop directly if using rand t, but used for FM class

model_fm = TimeConditionalUNet(num_hiddens=hidden_dim).to(device)
fm_wrapper = TimeConditionalFM(model_fm, num_ts=num_ts).to(device)
optimizer = optim.Adam(model_fm.parameters(), lr=lr)
scheduler = optim.lr_scheduler.ExponentialLR(optimizer, gamma=0.9)

losses = []

print("Training Time-Conditioned Flow Matching...")
for epoch in range(num_epochs):
    model_fm.train()
    batch_losses = []

    for images, _ in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
        images = images.to(device)

        loss = fm_wrapper(images) # Calls time_fm_forward

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        batch_losses.append(loss.item())

    scheduler.step()
    avg_loss = np.mean(batch_losses)
    losses.append(avg_loss)
    print(f"Epoch {epoch+1}, Loss: {avg_loss:.4f}")

    # Visualization
    if (epoch + 1) in [1, 5, 10]:
        samples = fm_wrapper.sample((28, 28), seed=42)

        fig, axs = plt.subplots(2, 8, figsize=(16, 4))
        for k in range(16):
            r, c = k // 8, k % 8
            axs[r, c].imshow(samples[k].cpu().squeeze(), cmap='gray')
            axs[r, c].axis('off')
        plt.suptitle(f"Time-Conditioned Samples Epoch {epoch+1}")
        plt.show()

# Plot Loss Curve
plt.plot(losses)
plt.title("Training Loss")
plt.show()

# Part 2.3: Sampling from the Time-conditioned UNet

In [ ]:
model_fm.eval()
fm_wrapper.eval()

print("Sampling from Time-Conditioned Model...")

seed = 114514

samples = fm_wrapper.sample(img_wh=(28, 28), seed=seed)

num_samples = samples.shape[0]
rows = 2
cols = num_samples // rows

plt.figure(figsize=(15, 5))
for i in range(num_samples):
    plt.subplot(rows, cols, i + 1)
    img_np = samples[i].permute(1, 2, 0).cpu().numpy()
    plt.imshow(img_np.squeeze(), cmap='gray')
    plt.axis('off')

plt.suptitle("Samples from Time-Conditioned Flow Matching")
plt.show()

# Part 2.4: Implementing a Class-conditioned UNet

In [ ]:
class ClassConditionalUNet(nn.Module):
    def __init__(self, in_channels: int = 1, num_classes: int = 10, num_hiddens: int = 64):
        super().__init__()

        # Base UNet Layers
        self.init_conv = Conv(in_channels, num_hiddens)
        self.down1 = DownBlock(num_hiddens, num_hiddens * 2)
        self.down2 = DownBlock(num_hiddens * 2, num_hiddens * 4)
        self.flatten = Flatten()

        # Conditioning Blocks
        # One-hot embedding dimension = num_classes
        self.fc1_t = FCBlock(1, num_hiddens * 4)
        self.fc1_c = FCBlock(num_classes, num_hiddens * 4)

        self.fc2_t = FCBlock(1, num_hiddens * 2)
        self.fc2_c = FCBlock(num_classes, num_hiddens * 2)

        self.unflatten = Unflatten(num_hiddens * 4)
        self.up1 = UpBlock(num_hiddens * 4, num_hiddens * 2)
        self.up2 = UpBlock(num_hiddens * 2, num_hiddens)
        self.out_conv = nn.Conv2d(num_hiddens, in_channels, kernel_size=3, padding=1)

        self.num_classes = num_classes

    def forward(self, x: torch.Tensor, c: torch.Tensor, t: torch.Tensor, mask: torch.Tensor | None = None) -> torch.Tensor:
        # c: (N,) int labels
        # mask: (N,) 0/1 mask for CFG training (1=keep, 0=drop)

        # Process Class Embeddings
        # Convert c to one-hot
        c_onehot = F.one_hot(c, num_classes=self.num_classes).float().to(x.device)

        if mask is not None:
            # mask (N,) -> (N, 1)
            c_onehot = c_onehot * mask.unsqueeze(1)

        # Encoding
        x1 = self.init_conv(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x_flat = self.flatten(x3)

        # Compute Embeddings
        t1 = self.fc1_t(t)
        c1 = self.fc1_c(c_onehot)

        t2 = self.fc2_t(t)
        c2 = self.fc2_c(c_onehot)

        # Reshape
        t1 = t1.view(-1, t1.shape[1], 1, 1)
        c1 = c1.view(-1, c1.shape[1], 1, 1)
        t2 = t2.view(-1, t2.shape[1], 1, 1)
        c2 = c2.view(-1, c2.shape[1], 1, 1)

        # unflatten = c1 * unflatten + t1
        x_unflat = self.unflatten(x_flat)
        x_unflat = c1 * x_unflat + t1

        # up1 = c2 * up1 + t2
        x_up1 = self.up1(x_unflat, x3)
        x_up1 = c2 * x_up1 + t2

        x_up2 = self.up2(x_up1, x2)
        out = self.out_conv(x_up2)
        return out

In [ ]:
def class_fm_forward(unet: ClassConditionalUNet, x_1: torch.Tensor, c: torch.Tensor, p_uncond: float, num_ts: int) -> torch.Tensor:
    unet.train()
    device = x_1.device
    B = x_1.shape[0]

    x_0 = torch.randn_like(x_1)
    t = torch.rand(B, device=device)
    t_img = t.view(-1, 1, 1, 1)
    x_t = (1 - t_img) * x_0 + t_img * x_1
    v_t = x_1 - x_0

    # mask = 1 (keep class), mask = 0 (drop class / null condition)
    # Bernoulli sampling
    mask = torch.bernoulli(torch.full((B,), 1 - p_uncond, device=device))

    v_pred = unet(x_t, c, t, mask)

    return F.mse_loss(v_pred, v_t)



In [ ]:
@torch.inference_mode()
def class_fm_sample(
    unet: ClassConditionalUNet,
    c: torch.Tensor,
    img_wh: tuple[int, int],
    num_ts: int,
    guidance_scale: float = 5.0,
    seed: int = 0
) -> torch.Tensor:
    unet.eval()
    torch.manual_seed(seed)
    device = next(unet.parameters()).device
    B = c.shape[0]

    # Start noise
    x = torch.randn(B, 1, img_wh[0], img_wh[1], device=device)
    timesteps = torch.linspace(0, 1, num_ts, device=device)
    dt = 1.0 / (num_ts - 1)

    caches = []

    for i in range(len(timesteps) - 1):
        t_curr = timesteps[i]
        t_batch = torch.full((B,), t_curr, device=device)

        # CFG Prediction
        v_cond = unet(x, c, t_batch, mask=torch.ones(B, device=device))

        v_uncond = unet(x, c, t_batch, mask=torch.zeros(B, device=device))

        # v = v_uncond + w * (v_cond - v_uncond)
        v_pred = v_uncond + guidance_scale * (v_cond - v_uncond)

        x = x + v_pred * dt
        # caches.append(x.cpu())

    return x.clamp(0, 1)

In [ ]:
class ClassConditionalFM(nn.Module):
    def __init__(
        self,
        unet: ClassConditionalUNet,
        num_ts: int = 300,
        p_uncond: float = 0.1,
    ):
        super().__init__()
        self.unet = unet
        self.num_ts = num_ts
        self.p_uncond = p_uncond

    def forward(self, x: torch.Tensor, c: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: (N, C, H, W) input tensor.
            c: (N,) int64 condition tensor.

        Returns:
            (,) loss.
        """
        return class_fm_forward(
            self.unet, x, c, self.p_uncond, self.num_ts
        )

    @torch.inference_mode()
    def sample(
        self,
        c: torch.Tensor,
        img_wh: tuple[int, int],
        guidance_scale: float = 5.0,
        seed: int = 0,
    ):
        return class_fm_sample(
            self.unet, c, img_wh, self.num_ts, guidance_scale, seed
        )

# Part 2.5 Training the Class-conditioned UNet

In [ ]:
# Hyperparams
lr = 1e-2
num_epochs = 10
hidden_dim = 64
num_ts = 50

model_cls = ClassConditionalUNet(num_hiddens=hidden_dim).to(device)
cls_wrapper = ClassConditionalFM(model_cls, num_ts=num_ts, p_uncond=0.1).to(device)
optimizer = optim.Adam(model_cls.parameters(), lr=lr)
scheduler = optim.lr_scheduler.ExponentialLR(optimizer, gamma=0.9)

losses = []

print("Training Class-Conditioned FM...")
for epoch in range(num_epochs):
    model_cls.train()
    batch_losses = []

    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
        images = images.to(device)
        labels = labels.to(device)

        loss = cls_wrapper(images, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        batch_losses.append(loss.item())

    scheduler.step()
    avg_loss = np.mean(batch_losses)
    losses.append(avg_loss)
    print(f"Epoch {epoch+1}, Loss: {avg_loss:.4f}")

    # Sample visualization
    if (epoch + 1) in [1, 5, 10]:
        # Generate 4 samples for each digit 0-9
        # Labels: [0,0,0,0, 1,1,1,1, ..., 9,9,9,9]
        c_eval = torch.arange(10, device=device).repeat_interleave(4)
        samples = cls_wrapper.sample(c_eval, (28, 28), guidance_scale=3.0, seed=42)

        fig, axs = plt.subplots(4, 10, figsize=(15, 6))
        for k in range(40):
            # map k to (row, col) in grid.
            # We want columns to be digits 0-9. Rows to be different samples.
            digit = k // 4
            instance = k % 4
            axs[instance, digit].imshow(samples[k].cpu().squeeze(), cmap='gray')
            axs[instance, digit].axis('off')
            if instance == 0:
                axs[instance, digit].set_title(str(digit))
        plt.suptitle(f"Class-Conditioned Samples Epoch {epoch+1}")
        plt.show()

# Part 2.6: Sampling from the Class-conditioned UNet

In [ ]:
model_cls.eval()
cls_wrapper.eval()

print("Sampling from Class-Conditioned Model with CFG...")

num_classes = 10
samples_per_class = 4
c = torch.arange(num_classes, device=device).repeat_interleave(samples_per_class)

guidance_scale = 5.0
seed = 123
img_wh = (28, 28)

generated_images = cls_wrapper.sample(
    c=c,
    img_wh=img_wh,
    guidance_scale=guidance_scale,
    seed=seed
)

plt.figure(figsize=(15, 6))

for k in range(generated_images.shape[0]):
    digit = k // samples_per_class
    instance = k % samples_per_class

    plot_idx = instance * num_classes + digit + 1

    plt.subplot(samples_per_class, num_classes, plot_idx)

    img_np = generated_images[k].permute(1, 2, 0).cpu().numpy()
    plt.imshow(img_np.squeeze(), cmap='gray')
    plt.axis('off')

    if instance == 0:
        plt.title(str(digit), fontsize=12, fontweight='bold')

plt.suptitle(f"Class-Conditioned Generation (CFG Scale = {guidance_scale})", y=1.02)
plt.tight_layout()
plt.show()